In [1]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [3]:
import random

# 固定随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # 设置固定种子


In [4]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# ----------------- CONFIGURATION -----------------
n_folds    = 10
n_trials   = 30
data_dir   = "./k_folds_model/embeddings"
batch_size = 256
num_epochs = 200
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------- MODEL DEFINITION -----------------
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        layers = []
        prev_dim = input_dim
        act = {
            'relu':     nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh':     nn.Tanh()
        }[activation]
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = self.net(x).squeeze(-1)
        return F.softplus(x)  # 输出 > 0

# ----------------- OPTUNA OBJECTIVE -----------------
def objective(trial):
    hidden_sizes      = trial.suggest_categorical('hidden_layer_sizes',
                            [(50,), (100,), (150,), (100, 50), (150, 100, 50)])
    activation        = trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh'])
    lr                = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    weight_decay      = trial.suggest_float('alpha', 1e-5, 1e-2, log=True)
    optimizer_choice  = trial.suggest_categorical('optimizer', ['sgd', 'adam'])

    fold_maes = []

    for fold in range(1, n_folds + 1):
        X_train = np.load(f"{data_dir}/train_fold_{fold}.npy")
        y_train = np.load(f"{data_dir}/train_labels_fold_{fold}.npy")
        X_val   = np.load(f"{data_dir}/val_fold_{fold}.npy")
        y_val   = np.load(f"{data_dir}/val_labels_fold_{fold}.npy")

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val   = scaler.transform(X_val)

        train_ds = TensorDataset(torch.from_numpy(X_train).float(),
                                 torch.from_numpy(y_train).float())
        val_ds = TensorDataset(torch.from_numpy(X_val).float(),
                               torch.from_numpy(y_val).float())
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        model = MLP(input_dim=X_train.shape[1],
                    hidden_sizes=hidden_sizes,
                    activation=activation).to(device)
        criterion = nn.MSELoss()
        optimizer = {
            'sgd': torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay),
            'adam': torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        }[optimizer_choice]

        try:
            for epoch in range(num_epochs):
                model.train()
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    preds = model(xb)
                    loss = criterion(preds, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            all_preds = []
            with torch.no_grad():
                for xb, _ in val_loader:
                    xb = xb.to(device)
                    batch_pred = model(xb).cpu().numpy()
                    all_preds.append(batch_pred)
            all_preds = np.concatenate(all_preds)

            mae = mean_absolute_error(y_val, all_preds)
            fold_maes.append(mae)

        except Exception:
            return np.inf

    return float(np.mean(fold_maes))

# ----------------- RUN OPTUNA -----------------
if __name__ == "__main__":
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials)

    print("\n=== Best Hyperparameters ===")
    print(study.best_params)
    print(f"Best Mean MAE: {study.best_value:.4f}")

[I 2025-05-16 10:03:19,831] A new study created in memory with name: no-name-5377a420-2056-4274-9b54-7e09acc72707
[I 2025-05-16 10:07:50,476] Trial 0 finished with value: 0.7171211242675781 and parameters: {'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate': 0.0015930522616241021, 'alpha': 0.001331121608073689, 'optimizer': 'adam'}. Best is trial 0 with value: 0.7171211242675781.
[I 2025-05-16 10:12:13,157] Trial 1 finished with value: 0.7098321914672852 and parameters: {'hidden_layer_sizes': (50,), 'activation': 'relu', 'learning_rate': 0.0016738085788752138, 'alpha': 2.621087878265438e-05, 'optimizer': 'adam'}. Best is trial 1 with value: 0.7098321914672852.
[I 2025-05-16 10:16:00,537] Trial 2 finished with value: 0.8029509782791138 and parameters: {'hidden_layer_sizes': (100,), 'activation': 'logistic', 'learning_rate': 0.00013492834268013249, 'alpha': 0.007025166339242158, 'optimizer': 'sgd'}. Best is trial 1 with value: 0.7098321914672852.
[I 2025-05-16 10:19:46,6


=== Best Hyperparameters ===
{'hidden_layer_sizes': (100,), 'activation': 'tanh', 'learning_rate': 0.0008316264993185375, 'alpha': 0.0008899223634459679, 'optimizer': 'sgd'}
Best Mean MAE: 0.6709


In [5]:
import os
import optuna
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold

# 配置
n_folds = 10
n_trials = 30
data_dir = './k_folds_model/embeddings'

# ========== 目标函数 ==========
def objective(trial):
    # 采样超参数
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
    }

    fold_maes = []

    for fold in range(1, n_folds + 1):
        # 读取数据
        X_train = np.load(os.path.join(data_dir, f'train_fold_{fold}.npy'))
        y_train = np.load(os.path.join(data_dir, f'train_labels_fold_{fold}.npy'))
        X_val = np.load(os.path.join(data_dir, f'val_fold_{fold}.npy'))
        y_val = np.load(os.path.join(data_dir, f'val_labels_fold_{fold}.npy'))

        model = XGBRegressor(n_jobs=-1, verbosity=0, **params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        mae = mean_absolute_error(y_val, y_pred)
        fold_maes.append(mae)

    return np.mean(fold_maes)  # ✅ 以 MAE 为优化目标


# ========== 启动 Optuna 搜索 ==========
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=n_trials)

# ========== 输出最优结果 ==========
best_params = study.best_params
best_score = study.best_value

print("\n=== Best XGBoost Hyperparameters via Optuna ===")
print(best_params)
print(f"10-Fold Mean MAE = {best_score:.4f}")

[I 2025-05-16 17:27:43,132] A new study created in memory with name: no-name-65977cc0-c1a9-4329-9ba7-362c080dfe26
[I 2025-05-16 17:30:08,387] Trial 0 finished with value: 0.7082074880599976 and parameters: {'n_estimators': 400, 'learning_rate': 0.17254716573280354, 'max_depth': 8, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.6624074561769746}. Best is trial 0 with value: 0.7082074880599976.
[I 2025-05-16 17:32:17,840] Trial 1 finished with value: 0.7287782430648804 and parameters: {'n_estimators': 200, 'learning_rate': 0.011900590783184251, 'max_depth': 9, 'subsample': 0.8803345035229626, 'colsample_bytree': 0.8832290311184181}. Best is trial 0 with value: 0.7082074880599976.
[I 2025-05-16 17:32:57,907] Trial 2 finished with value: 0.7006813287734985 and parameters: {'n_estimators': 100, 'learning_rate': 0.18276027831785724, 'max_depth': 8, 'subsample': 0.7637017332034828, 'colsample_bytree': 0.6727299868828402}. Best is trial 2 with value: 0.7006813287734985.
[I 2025-05-16 1


=== Best XGBoost Hyperparameters via Optuna ===
{'n_estimators': 600, 'learning_rate': 0.032962537792065316, 'max_depth': 9, 'subsample': 0.9318436391852422, 'colsample_bytree': 0.9198022055018699}
10-Fold Mean MAE = 0.6818


In [6]:
import os
import optuna
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

# 配置
n_folds = 10
n_trials = 30
data_dir = './k_folds_model/embeddings'

# ========== 目标函数 ==========
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'objective': 'poisson',         # ✅ 强制正输出
        'n_jobs': -1,
        'verbose': -1
    }

    fold_maes = []

    for fold in range(1, n_folds + 1):
        # 读取数据
        X_train = np.load(os.path.join(data_dir, f'train_fold_{fold}.npy'))
        y_train = np.load(os.path.join(data_dir, f'train_labels_fold_{fold}.npy'))
        X_val = np.load(os.path.join(data_dir, f'val_fold_{fold}.npy'))
        y_val = np.load(os.path.join(data_dir, f'val_labels_fold_{fold}.npy'))

        # Poisson 回归要求标签必须大于 0
        if np.any(y_train <= 0) or np.any(y_val <= 0):
            return np.inf

        model = LGBMRegressor(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        mae = mean_absolute_error(y_val, y_pred)
        fold_maes.append(mae)

    return np.mean(fold_maes)  # ✅ 使用 MAE 作为优化目标

# ========== 启动 Optuna 搜索 ==========
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=n_trials)

# ========== 输出最优结果 ==========
best_params = study.best_params
best_score = study.best_value

print("\n=== Best LightGBM Hyperparameters via Optuna (Poisson + MAE) ===")
print(best_params)
print(f"10-Fold Mean MAE = {best_score:.4f}")



[I 2025-05-16 18:45:01,442] A new study created in memory with name: no-name-5022d1a7-4728-471c-b18e-2627acf1f0af
[I 2025-05-16 18:45:27,815] Trial 0 finished with value: 0.671222476723558 and parameters: {'n_estimators': 400, 'learning_rate': 0.17254716573280354, 'max_depth': 10, 'num_leaves': 68, 'feature_fraction': 0.6624074561769746, 'bagging_fraction': 0.662397808134481}. Best is trial 0 with value: 0.671222476723558.
[I 2025-05-16 18:45:39,397] Trial 1 finished with value: 0.6788303879384187 and parameters: {'n_estimators': 100, 'learning_rate': 0.13394334706750485, 'max_depth': 9, 'num_leaves': 77, 'feature_fraction': 0.608233797718321, 'bagging_fraction': 0.9879639408647978}. Best is trial 0 with value: 0.671222476723558.
[I 2025-05-16 18:46:11,327] Trial 2 finished with value: 0.6949814712235833 and parameters: {'n_estimators': 900, 'learning_rate': 0.018891200276189388, 'max_depth': 4, 'num_leaves': 34, 'feature_fraction': 0.7216968971838151, 'bagging_fraction': 0.80990257265


=== Best LightGBM Hyperparameters via Optuna (Poisson + MAE) ===
{'n_estimators': 900, 'learning_rate': 0.0507925407083067, 'max_depth': 11, 'num_leaves': 90, 'feature_fraction': 0.8927667771131523, 'bagging_fraction': 0.838817870015776}
10-Fold Mean MAE = 0.6693
